In [1]:
import numpy as np
import pandas as pd
import psycopg2
import datetime

In [2]:
## add proper credenials!

conn = psycopg2.connect("dbname=kkdj20_test user=sebastian host=localhost password=123456")
#conn = psycopg2.connect("dbname=kraladies user=sebastian host=localhost password=123456")

cur = conn.cursor()

In [40]:

for EVENT_ID in [4,5,6,7,8,9]:

    _bid=[]
    _bsum=[]

    try:
        # select event_id with of <EVENT_NAME>
        cur.execute("SELECT id FROM baskets_event AS be WHERE be.event_name LIKE %s", (EVENT_NAME,))
        res = cur.fetchall()
        if res:
            event_id = int(res[0][0])
    except:
        event_id = EVENT_ID

    total_sum=0

    #if res:
    #    event_id = int(res[0][0])
        # select all baskets in <EVENT_NAME>
    cur.execute('''SELECT bb.id
                        , bb.created
                        , bb.last_modified
                FROM baskets_basket as bb 
                WHERE bb.event_id=%s 
                ORDER BY bb.created
                ''',(event_id,))
    res = cur.fetchall()
    print (f"\n\tThere are a total of {len(res)} items assigned to event_id={event_id}\n")

    for bid in res:
        basket_id = bid[0]
        b_c=datetime.datetime.strftime(bid[1], "%d/%m %H:%M:%S")
        b_lm=datetime.datetime.strftime(bid[2], "%d/%m %H:%M:%S")
        b_dt = (bid[2] - bid[1]).total_seconds()/60.
        #print (f"Basket No{basket_id} (created:{b_c}; last_mod:{b_lm}; dt:{b_dt:.1f}min)")
        cur.execute('''SELECT bi.id
                            , bi.price
                            , bi."vendorID_id" AS vid
                            , bv.vendor_number as vnr
                            , bi.created
                        FROM baskets_item AS bi 
                        JOIN baskets_vendor as bv 
                            ON bv.id=bi."vendorID_id"
                        WHERE bi.basket_id=%s
                        ORDER BY bi.created
                    ''',(basket_id,))

        r2 = cur.fetchall()
        _sum=0
        for iid in r2:
            item_id=iid[0]
            price=iid[1]
            vendor_id=iid[2]
            vendor_nr=iid[3]
            #item_created=iid[4]
            item_created=datetime.datetime.strftime(iid[4], "%d/%m %H:%M:%S")
            _sum+=price
            total_sum +=price
            ##print ("  Item{}: {:.2f} Euro  (Vendor #{}) ({})".format(item_id, price, vendor_nr, item_created))
        ##print ("   ...Total = {:.2f} Euro".format(_sum))
        _bid.append(bid)
        _bsum.append(_sum)

    print ("Gesamt Total = {:.2f} Euro".format(total_sum))

    _bid=np.array(_bid)
    _bsum=np.array(_bsum)
    
    print ("Koerbe fuer mehr als  25 Euro: ", len(_bsum[(_bsum>25)]))
    print ("Koerbe fuer mehr als  50 Euro: ", len(_bsum[(_bsum>50)]))
    print ("Koerbe fuer mehr als  75 Euro: ", len(_bsum[(_bsum>75)]))
    print ("Koerbe fuer mehr als 100 Euro: ", len(_bsum[(_bsum>100)]))


	There are a total of 360 items assigned to event_id=4

Gesamt Total = 9279.15 Euro
Koerbe fuer mehr als  25 Euro:  154
Koerbe fuer mehr als  50 Euro:  43
Koerbe fuer mehr als  75 Euro:  13
Koerbe fuer mehr als 100 Euro:  2

	There are a total of 406 items assigned to event_id=5

Gesamt Total = 10871.80 Euro
Koerbe fuer mehr als  25 Euro:  176
Koerbe fuer mehr als  50 Euro:  48
Koerbe fuer mehr als  75 Euro:  17
Koerbe fuer mehr als 100 Euro:  8

	There are a total of 398 items assigned to event_id=6

Gesamt Total = 11921.80 Euro
Koerbe fuer mehr als  25 Euro:  198
Koerbe fuer mehr als  50 Euro:  70
Koerbe fuer mehr als  75 Euro:  18
Koerbe fuer mehr als 100 Euro:  9

	There are a total of 447 items assigned to event_id=7

Gesamt Total = 12176.60 Euro
Koerbe fuer mehr als  25 Euro:  184
Koerbe fuer mehr als  50 Euro:  69
Koerbe fuer mehr als  75 Euro:  25
Koerbe fuer mehr als 100 Euro:  10

	There are a total of 394 items assigned to event_id=8

Gesamt Total = 12632.70 Euro
Koerbe fue